In [1]:
%cd ..

/home/bhkuser/bhklab/katy/readii_2_roqc


In [2]:
import itertools
from pathlib import Path

import click
import pandas as pd
import SimpleITK as sitk
from damply import dirs
from imgtools.io.writers.nifti_writer import NIFTIWriter, NiftiWriterIOError
from tqdm import tqdm

from readii.image_processing import alignImages, flattenImage
from readii.io.loaders import loadImageDatasetConfig
from readii.negative_controls_refactor import NegativeControlManager
from readii.process.config import get_full_data_name
from readii.utils import logger
from readii_2_roqc.utils.metadata import get_masked_image_metadata
from readii_2_roqc.readii.make_negative_controls import get_readii_settings


dataset = 'NSCLC-Radiomics_test'
overwrite = False
seed = 10

# get path to dataset config directory
config_dir_path = dirs.CONFIG / 'datasets'

# Load in dataset configuration settings from provided dataset name
dataset_config = loadImageDatasetConfig(dataset, config_dir_path)

dataset_name = dataset_config['DATASET_NAME']
full_data_name = get_full_data_name(config_dir_path / dataset)

# Extract READII settings from config file
regions, permutations, _crop = get_readii_settings(dataset_config)

# Set up negative control manager with settings from config
manager = NegativeControlManager.from_strings(
negative_control_types=permutations,
region_types=regions,
random_seed=seed
)

# Get path to the images output by med-imagetools autopipeline run
mit_images_dir_path = dirs.PROCDATA / full_data_name / 'images' /f'mit_{dataset_name}'

# Load in mit_index file
dataset_index = pd.read_csv(Path(mit_images_dir_path, f'mit_{dataset_name}_index-simple.csv'))

# Get just the rows with the desired image and mask modalities and ROIs specified in the dataset config
image_modality = dataset_config["MIT"]["MODALITIES"]["image"]
mask_modality = dataset_config["MIT"]["MODALITIES"]["mask"]
masked_image_index = get_masked_image_metadata(dataset_index = dataset_index,
                                            dataset_config = dataset_config,
                                            image_modality = image_modality,
                                            mask_modality = mask_modality)

# Set up directory to save out the negative controls
readii_image_dir = mit_images_dir_path.parent / f'readii_{dataset_name}'

# Check for index file existence and overwrite status to determine if continuing to negative control creation
readii_index_file = readii_image_dir / f'readii_{dataset_name}_index.csv'

In [4]:
if readii_index_file.exists() and not overwrite: 
    # Load in readii index and check:
    # 1. if all negative controls requested have been extracted
    # 2. for all of the patients
    readii_index = pd.read_csv(readii_index_file)

    # Get list of patients that have already been processed and what has been requested based on the dataset index
    processed_samples = set(readii_index['PatientID'].to_list())
    requested_samples = set(dataset_index['PatientID'].to_list())

    processed_image_types = {itype for itype in readii_index[['Permutation', 'Region']].itertuples(index=False, name=None)}
    requested_image_types = {itype for itype in itertools.product([permutation.name() for permutation in manager.negative_control_strategies],
                                                                [region.name() for region in manager.region_strategies])}

In [7]:
processed_image_types

{('randomized', 'full'),
 ('randomized', 'non_roi'),
 ('randomized', 'roi'),
 ('shuffled', 'full'),
 ('shuffled', 'non_roi'),
 ('shuffled', 'roi')}

In [8]:
processed_samples

{'LUNG1-001', 'LUNG1-002'}

In [11]:
readii_index[['Permutation', 'Region']].itertuples(index=False, name=None)

In [13]:
readii_index[readii_index['PatientID'].isin(processed_samples)]

,ImageID_mask,PatientID,Permutation,Region,dir_original_image,dirname_mask,filepath,saved_time
0,GTV__[GTV-1],LUNG1-002,shuffled,full,LUNG1-002_0001/CT_23261228,RTSTRUCT_43245931,LUNG1-002_0001/CT_23261228/RTSTRUCT_43245931_G...,2025-07-30:19:23:43
1,GTV__[GTV-1],LUNG1-002,shuffled,roi,LUNG1-002_0001/CT_23261228,RTSTRUCT_43245931,LUNG1-002_0001/CT_23261228/RTSTRUCT_43245931_G...,2025-07-30:19:23:45
2,GTV__[GTV-1],LUNG1-002,shuffled,non_roi,LUNG1-002_0001/CT_23261228,RTSTRUCT_43245931,LUNG1-002_0001/CT_23261228/RTSTRUCT_43245931_G...,2025-07-30:19:23:49
3,GTV__[GTV-1],LUNG1-001,randomized,full,LUNG1-001_0000/CT_63382046,RTSTRUCT_35578236,LUNG1-001_0000/CT_63382046/RTSTRUCT_35578236_G...,2025-07-30:19:24:40
4,GTV__[GTV-1],LUNG1-001,randomized,roi,LUNG1-001_0000/CT_63382046,RTSTRUCT_35578236,LUNG1-001_0000/CT_63382046/RTSTRUCT_35578236_G...,2025-07-30:19:24:43
5,GTV__[GTV-1],LUNG1-001,randomized,non_roi,LUNG1-001_0000/CT_63382046,RTSTRUCT_35578236,LUNG1-001_0000/CT_63382046/RTSTRUCT_35578236_G...,2025-07-30:19:24:49


In [14]:
masked_image_index

,ImageID,Modality,PatientID,ReferencedSeriesUID,SampleID,SampleNumber,SeriesInstanceUID,StudyInstanceUID,class,direction,...,ndim,nvoxels,origin,roi_key,saved_time,size,spacing,std,sum,variance
0,CT,CT,LUNG1-001,NaN,LUNG1-001_0000,0,1.3.6.1.4.1.32722.99.99.2989917765213423750108...,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,Scan,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",...,3,35127296,"(-249.51171875, -460.51171875, -681.5)",NaN,2025-07-22:14:16:16,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)",426.958598,-2.604295e+10,182293.644053
2,CT,CT,LUNG1-002,NaN,LUNG1-002_0001,1,1.3.6.1.4.1.32722.99.99.2329880015517990803358...,1.3.6.1.4.1.32722.99.99.2037150038059966416957...,Scan,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",...,3,29097984,"(-250.112, -250.112, -133.4)",NaN,2025-07-22:14:16:15,"(512, 512, 111)","(0.977, 0.977, 3.0)",431.016943,-2.197104e+10,185775.604855
0,GTV__[GTV-1],RTSTRUCT,LUNG1-001,1.3.6.1.4.1.32722.99.99.2989917765213423750108...,LUNG1-001_0000,0,1.3.6.1.4.1.32722.99.99.2279381215866080725084...,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,Mask,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",...,3,35127296,"(-249.51171875, -460.51171875, -681.5)",GTV,2025-07-22:14:16:22,"(512, 512, 134)","(0.9765625, 0.9765625, 3.0)",0.039992,5.627100e+04,0.001599
1,GTV__[GTV-1],RTSTRUCT,LUNG1-002,1.3.6.1.4.1.32722.99.99.2329880015517990803358...,LUNG1-002_0001,1,1.3.6.1.4.1.32722.99.99.2432675512669112458302...,1.3.6.1.4.1.32722.99.99.2037150038059966416957...,Mask,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",...,3,29097984,"(-250.112, -250.112, -133.4)",GTV,2025-07-22:14:16:18,"(512, 512, 111)","(0.977, 0.977, 3.0)",0.065496,1.253640e+05,0.004290
